# 02. Logistic Regression - Titanic Survival**Topic:** Predict whether a Titanic passenger survived based on their information.**Dataset:** Titanic (built into seaborn).**What you will do:**1. Load and clean the Titanic dataset2. Engineer simple features3. Train a logistic regression model4. Evaluate with accuracy, precision, recall, F1, ROC-AUC5. Plot the ROC curve and confusion matrix

In [ ]:
# Run this cell first if running locally. In Google Colab everything is pre-installed.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)np.random.seed(42)

## 1. Load the data

In [ ]:
df = sns.load_dataset("titanic")print(df.shape)df.head()

In [ ]:
print(df.isnull().sum().sort_values(ascending=False))

## 2. Clean and prepare features

In [ ]:
# Pick useful columnsdf = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()# Engineer family size + alone flagdf["family_size"] = df["sibsp"] + df["parch"] + 1df["is_alone"] = (df["family_size"] == 1).astype(int)# Fill missing valuesdf["age"].fillna(df["age"].median(), inplace=True)df["embarked"].fillna(df["embarked"].mode()[0], inplace=True)# One-hot encode categoricalsdf = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True)df.head()

## 3. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.linear_model import LogisticRegressionX = df.drop(columns=["survived"])y = df["survived"]X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print(X_train.shape, X_test.shape)

## 4. Logistic Regression Pipeline

In [ ]:
pipe = Pipeline([    ("scaler", StandardScaler()),    ("model",  LogisticRegression(max_iter=1000)),])pipe.fit(X_train, y_train)y_pred  = pipe.predict(X_test)y_proba = pipe.predict_proba(X_test)[:, 1]

## 5. Evaluation

In [ ]:
from sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score,    roc_auc_score, confusion_matrix, classification_report,    roc_curve,)print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")print(f"Precision: {precision_score(y_test, y_pred):.3f}")print(f"Recall:    {recall_score(y_test, y_pred):.3f}")print(f"F1:        {f1_score(y_test, y_pred):.3f}")print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.3f}")print()print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

In [ ]:
# Confusion matrixcm = confusion_matrix(y_test, y_pred)sns.heatmap(cm, annot=True, fmt="d", cmap="Greys",            xticklabels=["Died", "Survived"], yticklabels=["Died", "Survived"])plt.xlabel("Predicted")plt.ylabel("Actual")plt.title("Confusion Matrix")plt.show()

In [ ]:
# ROC curvefpr, tpr, thresholds = roc_curve(y_test, y_proba)plt.figure(figsize=(8, 6))plt.plot(fpr, tpr, color="black", lw=2, label=f"AUC = {roc_auc_score(y_test, y_proba):.2f}")plt.plot([0, 1], [0, 1], color="gray", linestyle="--")plt.xlabel("False Positive Rate")plt.ylabel("True Positive Rate")plt.title("ROC Curve")plt.legend()plt.show()

## 6. Inspect coefficients

In [ ]:
coef_df = pd.DataFrame({    "Feature": X.columns,    "Coefficient": pipe.named_steps["model"].coef_[0],}).sort_values("Coefficient", key=abs, ascending=False)print(coef_df)

## 7. Exercises1. **Adjust the threshold** from the default 0.5 and watch precision/recall change.2. **Try a different solver** like `solver="liblinear"` or `"saga"`.3. **Add L1 regularization** with `penalty="l1"` and `solver="liblinear"`.4. **Add the passenger title** as a feature (extract from the original `name` column - load the raw Kaggle dataset).5. **Plot a calibration curve** to check if probabilities are well-calibrated.